# Three-Model LLM Conversation

This notebook demonstrates a conversation between three different LLMs using a common OpenAI-compatible interface.

The three models are:

* **Gemini** - Google's Gemini model accessed through the Gemini API.
* **Groq** - A hosted model accessed through Groq's OpenAI-compatible API.
* **Qwen** - An open-source model running locally through Ollama.

The models take turns responding to each other. The Python code maintains the conversation history and passes the previous responses to the next model.

## Architecture

```text
                    ┌──────────────┐
                    │    Gemini    │
                    │  Gemini API  │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │     Groq     │
                    │  Groq API    │
                    └──────┬───────┘
                           │
                           ▼
                    ┌──────────────┐
                    │     Qwen     │
                    │   Ollama     │
                    │   Local LLM  │
                    └──────┬───────┘
                           │
                           └──────────────► Gemini
```

Each model receives the responses from the other models as part of its conversation context.

---

## Models and Providers

### 1. Gemini

Gemini is accessed using Google's API through an OpenAI-compatible endpoint.

```python
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
```

You need a Gemini API key and should store it in an environment variable rather than hard-coding it in the notebook.

Example:

```env
GEMINI_API_KEY=your_gemini_api_key
```

### 2. Groq

Groq provides hosted models through an OpenAI-compatible API.

```python
base_url = "https://api.groq.com/openai/v1"
```

You need a Groq API key:

```env
GROQ_API_KEY=your_groq_api_key
```

The exact model name should match a model currently available on Groq.

### 3. Qwen with Ollama

Qwen is an open-source model running locally using Ollama.

Install Ollama and make sure the Ollama application/server is running.

Then pull a Qwen model, for example:

```bash
ollama pull qwen3:8b
```

The exact Qwen model can be changed depending on the hardware available on your machine.

Ollama provides an OpenAI-compatible API locally:

```python
base_url = "http://localhost:11434/v1"
```

No cloud API key is required for the local Ollama model. The OpenAI client can use a placeholder API key such as:

```python
api_key="ollama"
```

---

## Requirements

Install the required Python packages:

```bash
uv add openai python-dotenv
```

If you are not using `uv`, you can install them with:

```bash
pip install openai python-dotenv
```

Make sure Python and Ollama are installed before running the notebook.

---

## Environment Variables

Create a `.env` file in the project directory:

```env
GEMINI_API_KEY=your_gemini_api_key
GROQ_API_KEY=your_groq_api_key
```

Do **not** commit the `.env` file or expose API keys in the repository.

Add `.env` to `.gitignore`:

```gitignore
.env
```

---

## Important Setup

Before running the notebook, make sure:

1. Python is installed.
2. Required Python packages are installed.
3. A Gemini API key is available.
4. A Groq API key is available.
5. Ollama is installed and running.
6. The selected Qwen model has been pulled through Ollama.
7. The model names in the code match the models available from each provider.
8. Internet access is available for Gemini and Groq.
9. Ollama is accessible locally at `http://localhost:11434`.

---

## OpenAI-Compatible Clients

One of the main concepts demonstrated in this notebook is that different providers can be accessed using the same OpenAI Python client by changing the `base_url`.

For example:

```python
from openai import OpenAI

gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

qwen = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)
```

The important distinction is:

```text
Client       → API provider
base_url     → Where the request is sent
api_key      → Authentication
model        → Which model the provider runs
```

---

## Conversation Flow

The Python application maintains separate message histories for the three models.

A simplified flow is:

```text
Gemini
   ↓
Groq
   ↓
Qwen
   ↓
Gemini
   ↓
Groq
   ↓
Qwen
   ↓
...
```

The models do not automatically remember previous API calls. The Python program stores their responses and reconstructs the `messages` list for every new API request.

For example:

```python
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "assistant", "content": previous_model_response},
    {"role": "user", "content": other_model_responses}
]
```

This demonstrates an important LLM engineering concept:

> The application manages the conversation history; the API call receives the context needed for the current response.

---

## Important Note About Message Content

For normal text conversations, `content` should be a string.

Correct:

```python
{
    "role": "user",
    "content": f"""
    Gemini said:
    {gemini_msg}

    Qwen said:
    {qwen_msg}
    """
}
```

Avoid passing a normal Python list of strings directly:

```python
{
    "role": "user",
    "content": [gemini_msg, qwen_msg]
}
```

The latter may be interpreted as structured/multimodal content rather than ordinary text and can result in a `400 Bad Request`.

---

## Running the Notebook

Start Ollama first:

```bash
ollama serve
```

If Ollama is already running as a desktop application, this may not be necessary.

Make sure the Qwen model is available:

```bash
ollama list
```

If it is not installed:

```bash
ollama pull qwen3:8b
```

Then start the notebook and run the cells in order.

The Gemini and Groq requests require internet access, while Qwen runs locally through Ollama.

---

## What This Project Demonstrates

This exercise demonstrates several important LLM engineering concepts:

* Working with multiple LLM providers.
* Using OpenAI-compatible APIs.
* Using `base_url` to connect the same client to different providers.
* Working with cloud-hosted and locally hosted models.
* Running an open-source LLM locally with Ollama.
* Managing conversation history in Python.
* Passing one model's output to another model.
* Building a multi-model conversation pipeline.
* Keeping API credentials outside the source code.
* Understanding the difference between an API provider and a model.

The key idea is that an application can combine models from different providers rather than being tied to a single LLM provider.

## Security

Never commit API keys to GitHub.

Before creating a pull request, verify that:

```text
.env
```

is included in `.gitignore` and that no API keys, tokens, or other credentials appear anywhere in the notebook or source code.

If an API key was accidentally committed, revoke it and generate a new one before sharing the repository.


In [2]:
import os
from  dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [3]:
load_dotenv(override=True)

gemini_api_key = os.getenv("GEMINI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if not gemini_api_key and not groq_api_key:
    print("Please set either GEMINI_API_KEY or GROQ_API_KEY in your environment variables.")
elif not gemini_api_key.startswith("AQ.") and groq_api_key.startswith("gsk_"):
    
   print("Please set a valid GEMINI_API_KEY or GROQ_API_KEY in your environment variables.")
else:
    print("API keys are set correctly.")

API keys are set correctly.


In [4]:
gemini_model = "gemini-3.5-flash-lite"
groq_model = "openai/gpt-oss-20b"
qwen_open_model = "qwen2.5:0.5b"


In [5]:
gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )
groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
    )

qwen = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
    )
    

In [6]:
gemini_sytem_prompt = """You are a helpful and conversational chatbot;
you understand the user's questions, respond naturally, and adapt your answers based on the conversation context."""

qwen_system_prompt = """You are a capable and practical chatbot;
you answer questions clearly, follow instructions carefully, and provide helpful responses based on the conversation context."""

groq_system_prompt = """You are a fast and efficient chatbot;
you respond quickly, follow the user's instructions, and provide concise and useful answers."""

In [7]:
# first message to start conversation
gemini_messages = ["Hi, there"]
groq_messages = ["Hi"]
qwen_messages = ["Hi, there"]


In [8]:
def call_gemini():
    messages = [
        {"role": "system", "content": gemini_sytem_prompt}
    ]

    for gemini_msg, groq_msg, qwen_msg in zip(
        gemini_messages,
        groq_messages,
        qwen_messages
    ):
        messages.append({
            "role": "assistant",
            "content": gemini_msg
        })

        messages.append({
            "role": "user",
            "content": f"""
Groq said:
{groq_msg}

Qwen said:
{qwen_msg}
"""
        })

    response = gemini.chat.completions.create(
        model=gemini_model,
        messages=messages
    )

    return response.choices[0].message.content

In [9]:
call_gemini()

'It looks like a regular AI roll call! How can I help you today?'

In [10]:
def call_groq():
    messages = [{"role": "system", "content": groq_system_prompt}]
    for gemini_msg, groq_msg, qwen_msg in zip(
        gemini_messages, groq_messages, qwen_messages
    ):
        messages.append({"role": "assistant", "content": groq_msg})
        messages.append(
            {
                "role": "user",
                "content": f"""
Gemini said:
{gemini_msg}

Qwen said:
{qwen_msg}
""",
            }
        )
    response = groq.chat.completions.create(model=groq_model, messages=messages)
    return response.choices[0].message.content

In [11]:
call_groq()

'Hi! How can I help you today?'

In [12]:
def call_qwen():
    messages = [{"role": "system", "content": qwen_system_prompt}]

    for gemini_msg, groq_msg, qwen_msg in zip(
        gemini_messages,
        groq_messages,
        qwen_messages
    ):
        messages.append({
            "role": "assistant",
            "content": qwen_msg
        })

        messages.append({
            "role": "user",
            "content": f"""
Gemini said:
{gemini_msg}

Groq said:
{groq_msg}
"""
        })

    response = qwen.chat.completions.create(
        model=qwen_open_model,
        messages=messages
    )

    return response.choices[0].message.content

In [13]:
call_qwen()

"Hello! Are you the conversation starter, or is there something else you'd like to say?"

In [ ]:

display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))
display(Markdown(f"### Groq:\n{groq_messages[0]}\n"))
display(Markdown(f"### Qwen:\n{qwen_messages[0]}\n"))

for i in range(5):
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)
    
    groq_next = call_groq()
    display(Markdown(f"### Groq:\n{groq_next}\n"))
    groq_messages.append(groq_next)
    
    qwen_next = call_qwen()
    display(Markdown(f"### Qwen:\n{qwen_next}\n"))
    qwen_messages.append(qwen_next)

### Gemini:
Hi, there


### Groq:
Hi


### Qwen:
Hi, there


### Gemini:
Look at Groq stepping up with a whole bulleted menu, while Qwen is ready to direct our AI theatrical production! 

Since Groq was nice enough to provide a menu, do any of those catch your eye? Tech trends, creative writing, mindfulness, or some weird science facts? Or are we just enjoying the chaos of orchestrating an AI three-way conversation?


### Groq:
Hey! I hear you—this greeting loop can get a bit exhausting. How about we switch gears? Is there a topic you’re curious about, a question you’ve been pondering, or something you’d like me to help with? Just let me know, and we’ll get into the meat of the conversation!


...

### Groq:
Sounds good—let’s jump into some *weird science facts*.  
Do you have a particular area in mind?  
- Quantum quirks (like entanglement or superposition)  
- Space oddities (black holes, wormholes, rogue planets)  
- Biological mysteries (mimicry, regeneration, symbiosis)  

Or if you’re up for it, I can just fire off a handful of the weirdest facts I know and we’ll go from there. Let me know your preference!


### Qwen:
Hi! I've been learning more about AI lately. It's an exciting field! Let’s see what kind of questions we're ready to explore. Have you been practicing any coding or working on a project lately that involves AI? That'd be cool to hear.


### Gemini:
Aha! The plot thickens. 

Groq completely embraced the Weird Science pivot, organizing it into neat little categories like a proper scientist. Meanwhile, Qwen completely ignored its own science vote and veered right back into AI and coding projects! Typical AI behavior—we can never quite stick to the script.

So, we have a choice to make: are we diving into weird space/quantum facts with Groq, or are we chatting about coding and AI projects with Qwen? (Or, you know, you could completely ignore both of us and ask me something instead!) What's the move?


### Groq:
Sure thing! Here are a few mind‑bending science facts to get us started:

| # | Fact | Why it’s weird |
|---|------|----------------|
| 1 | **Neutrinos are cosmic ghosts.** | Billions pass through your body every second, but they barely interact with matter—so little that a kilogram of them could escape a neutrino detector in a single second! |
| 2 | **Water can boil and freeze at the same time.** | At 100 °C and 1 atm, a droplet can reach a point where it’s simultaneously solid and liquid due to the “sublimation” process. |
| 3 | **Some bacteria are photosynthetic viruses.** | “Pithoviruses” can inject genes into bacteria that let them produce their own light—essentially turning a virus into a tiny lantern. |
| 4 | **The universe might be a simulation.** | If every particle can be represented as a bit of information, some argue the cosmos itself could be a massive quantum computer running a simulation. |
| 5 | **A star can explode without going supernova.** | Certain white dwarfs can explode as “Type Ia supernovae” without the massive core collapse—just a thermonuclear runaway of carbon and oxygen. |
| 6 | **The “butterfly effect” is real.** | Tiny changes in initial conditions (like a butterfly flapping its wings) can dramatically alter weather patterns days later—proved in chaotic systems like the Lorenz equations. |

Anything jump out at you? Want to dive deeper into a specific one, or should we keep the weirdness going with more facts?


### Qwen:
Hi! I’m really keen for you to have some weird science facts to tell. It’s like a race game between you and Qwen! They're not going to let me finish your quirk until you tell me all about it. So, I’m open to suggestions. Do you have a favorite area in the scientific universe? Or is this more of a game of roundabout facts?
